# Synthetic narrative (patient, trial) relevance pairs

Manufactures the right-distribution data KZ never was: narrative patient notes tied to real trials with
**graded relevance labels** (`rel=2` eligible · `rel=1` on-topic-but-ineligible · we generate only rel≥1;
rel=0 comes free from the consumers' dense mining). Feeds `finetune_monot5_ct_v7` (§15/§16). Claude is used
for **data generation only**; no closed model touches the inference path.

## ▶ How to run — the phase map

Cells are grouped into numbered **PHASES**. Run top-to-bottom; each phase header states cost / hardware /
whether it resumes. **Start with Phase 0–2 (free, no API), then run the STATE CHECK cell to see where you
are** before spending on API/GPU.

| Phase | What | Cost | HW | Resumes? |
|---|---|---|---|---|
| 0 | Setup — install, API key, mount Drive | — | CPU | — |
| 1 | Config & registries — **the only knobs you flip** | — | CPU | — |
| — | **STATE CHECK** — reads Drive, prints exactly where you are | — | CPU | — |
| 2 | Build specs — corpus → trial pool → specs → prompts (no API) | — | CPU | — |
| 3 | Diagnostic + pilot — 60 pairs, read the reports before the full run | ~$0.20 | CPU | — |
| 4 | Full generation | ~$30–50 | CPU | ✅ by `syn_id` |
| 5 | Verification + fidelity report | ~$5 | CPU | ✅ by `syn_id` |
| 6 | Relevance gate — **disabled** (documented negative result) | — | — | — |
| 7 | Difficulty diagnostic + curation → writes the training file | — | **GPU** | — |
| 8 | Adopt → `synthetic_pairs.jsonl` (what the consumers read) | — | CPU | — |

**The knobs** all live in Phase 1's `SynthConfig` — change one key, everything downstream re-tags and
re-routes: `gen_model` (which generator), `gen_prompt` (v1 / v2_strict / **v3_implicit** = harder synth),
`verify_model`. Deeper toggles are tagged `# ⚙️ KNOB` where they live (relevance gate off; difficulty
source auto-detects pilot-vs-file). `cfg.tag()` stamps every output file, so prompt/model variants never
collide — A/B is a one-key change.

## PHASE 0 — Setup  ·  install, API key, mount Drive

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q anthropic tqdm datasets

In [ ]:
import os
os.environ['ANTHROPIC_API_KEY'] = ''  # ⚙️ paste your key (never commit; rotate if leaked)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## PHASE 1 — Config & registries  ·  the only knobs you flip

**Registries** (`MODELS` / `*_PROMPTS`) are the menu; **`SynthConfig`** picks from it by key. Add a variant
= add a registry entry + point `SynthConfig` at its key. Everything downstream reads through `cfg`, so
nothing else changes. Generator prompts: `v1` (baseline) · `v2_strict` (more explicit = easier) ·
`v3_implicit` (**harder synth** — bans criterion-echoing, forces inference; the §16 lever).

In [ ]:
# ── Model registry ───────────────────────────────────────────────────────────
MODELS = {
    'sonnet':  'claude-sonnet-4-6',   # default generator: cheap, fast, low refusal rate on clinical text
    'sonnet5': 'claude-sonnet-5',     # newer sonnet
    'haiku':   'claude-haiku-4-5',    # independent verifier (different family from the generator)
    'opus':    'claude-opus-4-8',     # stronger generator / QA re-check
    'fable':   'claude-fable-5',      # most capable — but see caveats below before using as the generator
}
# ⚠️ Claude Fable 5 (`fable`) caveats for THIS task (bulk clinical patient-note generation):
#   1. Safety classifiers can DECLINE life-sciences requests → HTTP 200 with stop_reason='refusal'
#      and an EMPTY content list. gen_patient/verify handle this (below), but a high refusal rate
#      will starve the run — watch the pilot's "refusal" count.
#   2. Requires 30-day data retention; a zero-data-retention org gets a 400 on EVERY request.
#   3. Pricing ($10/$50 per MTok) is steep for ~6k generations.
# For a large generation run, `sonnet` (or `sonnet5`) is the pragmatic generator. Reserve `fable`
# for a small high-quality slice, and if you use it, enable server-side fallbacks (see the diagnostic
# cell) so a refusal is transparently re-served instead of dropping the pair.

# ── Generator system prompts ─────────────────────────────────────────────────
GEN_PROMPTS = {
'v1': """You are generating synthetic patient notes for clinical trial matching research.

You are given a clinical CONDITION and a set of eligibility criteria from a real trial for that condition, each with a TARGET assessment. Write ONE realistic patient note such that a clinical expert reading it would (a) conclude the patient clearly HAS the given condition, and (b) reach exactly the target assessment for every listed criterion.

Target semantics:
- included: the note contains decisive evidence the patient MEETS this inclusion criterion
- not_included: the note contains decisive evidence the patient does NOT meet this inclusion criterion
- excluded: the note contains decisive evidence this exclusion criterion APPLIES to the patient
- not_excluded: the note contains decisive evidence this exclusion criterion does NOT apply

Rules:
1. The patient must clearly have the stated CONDITION — this is not optional. The note is a patient who presents with / is being worked up for / is under care for that condition.
2. Evidence may be implicit but must be decisive — e.g. "walks two miles daily" suffices for a performance-status criterion. A careful clinician must land on the target label, not merely find it plausible.
3. Add realistic unrelated detail (demographics, meds, social history, an incidental comorbidity) so the note does not read as constructed around the criteria.
4. Keep the note internally consistent and clinically plausible for the given age and sex.
5. If the target set is clinically contradictory or cannot be realized in one coherent patient who has the condition, respond with {"feasible": false} and nothing else.

Respond with JSON only, no markdown fences:
{"feasible": true, "patient_note": "...", "evidence": [{"idx": 0, "evidence": "short quote from the note"}]}""",

# v2_strict: same contract, harder on decisiveness + anti-boilerplate. A/B against v1 via the canary.
'v2_strict': """You are generating synthetic patient notes for clinical trial matching research.

You are given a clinical CONDITION and eligibility criteria from a real trial, each with a TARGET assessment. Write ONE realistic patient note so that an EXPERT ADJUDICATOR would independently reach exactly the target assessment for every criterion, with NO benefit of the doubt.

Target semantics:
- included / not_included: decisive evidence the patient MEETS / does NOT meet this inclusion criterion
- excluded / not_excluded: decisive evidence this exclusion criterion APPLIES / does NOT apply

Hard rules:
1. The patient must UNAMBIGUOUSLY have the CONDITION (named diagnosis or pathognomonic findings).
2. Every target must be supported by a SPECIFIC datum (a value, date, med, or event) — never vague adjectives. If the honest note would leave a target ambiguous, you have not satisfied it.
3. Vary structure and vocabulary run-to-run; do not reuse template sentences.
4. Internally consistent and plausible for the age/sex.
5. If the target set is contradictory, respond with {"feasible": false} and nothing else.

Respond with JSON only, no fences:
{"feasible": true, "patient_note": "...", "evidence": [{"idx": 0, "evidence": "short quote"}]}""",

# v3_implicit: the HARDER-SYNTH prompt (the §16 lever). The v1/v2 notes lexically ECHO the trial's
# eligibility text, so base monoT5 finds trivial overlap -> ~90% easier than real TREC positives ->
# low learning signal -> caps the recipe. Real TREC patients describe the same clinical reality in
# THEIR OWN vocabulary, forcing inference. This prompt bans criterion-echoing and encodes each target
# through clinical facts a reader must INFER. Opposite of v2_strict. Difficulty gate is the arbiter.
'v3_implicit': """You are generating synthetic patient notes for clinical trial matching research.

You are given a clinical CONDITION and eligibility criteria from a real trial, each with a TARGET assessment. Write ONE realistic patient note. A clinical expert reading it must (a) conclude the patient clearly HAS the condition, and (b) reach exactly the target assessment for every criterion — BY INFERENCE from the clinical picture, not because you restated the criterion.

Target semantics:
- included / not_included: decisive evidence the patient MEETS / does NOT meet this inclusion criterion
- excluded / not_excluded: decisive evidence this exclusion criterion APPLIES / does NOT apply

CRITICAL — write like a REAL patient note, NOT a checklist against the criteria:
1. Encode each target through the patient's actual clinical reality — history of present illness, past medical/surgical history, exam findings, vitals, labs, medications, functional/social status — in the natural vocabulary a clinician uses for THIS patient.
2. Do NOT echo or paraphrase the criterion's wording, and do NOT name the criterion or its category. The reader must INFER each assessment from the facts. Examples:
   - performance-status inclusion -> "works full-time as a roofer, hikes on weekends" (NEVER "good performance status" / "ECOG 1").
   - "no prior chemotherapy" -> a treatment history managed by surgery and radiation alone (NEVER "chemotherapy-naive").
   - "eGFR > 60" -> a normal creatinine in a patient with no renal history (let the reader infer adequate renal function).
3. Evidence must still be DECISIVE: a careful clinician lands on the target label with confidence. Implicit does NOT mean vague or omitted — it means shown through clinical facts rather than stated as a conclusion.
4. Add realistic unrelated detail so the note reads as a real patient, not one built around the criteria.
5. Internally consistent and clinically plausible for the age and sex.
6. If the target set cannot be realized in one coherent patient who has the condition, respond with {"feasible": false} and nothing else.

Respond with JSON only, no markdown fences:
{"feasible": true, "patient_note": "...", "evidence": [{"idx": 0, "evidence": "the clinical fact(s) in the note that imply this target"}]}""",
}

# ── Topicality-check prompts (verifier) ──────────────────────────────────────
TOPIC_PROMPTS = {
'v1': """You are a clinical assessor. Given a patient note and a condition name, decide whether the patient described has (or is being evaluated/treated for) that condition. Answer with exactly one word: yes or no.""",
}

# ── Per-criterion blind-verify prompts (verifier) ────────────────────────────
VERIFY_PROMPTS = {
'v1': """You are a clinical trial eligibility assessor.
Given a patient description and a single eligibility criterion, determine whether the criterion applies.

Respond with exactly one label and nothing else:
- included          (patient meets this inclusion criterion)
- not_included      (patient does not meet this inclusion criterion)
- excluded          (this exclusion criterion applies to the patient)
- not_excluded      (this exclusion criterion does not apply to the patient)
- not_enough_information  (cannot determine from the patient description)""",
}

# ── Whole-trial relevance gate prompts (verifier) ────────────────────────────
# Catches the categorical mislabels the per-criterion check misses: a patient who satisfies one
# cherry-picked criterion but is the WRONG kind of patient for the trial (wrong anatomy/population/
# core diagnosis). Deliberately biased toward "yes" so only categorical mismatches are dropped —
# an on-topic-but-ineligible patient (a valid rel=1) must PASS.
RELEVANCE_PROMPTS = {
'v1': """You are screening whether a patient is a plausible candidate for a clinical trial — a COARSE relevance check, not a fine eligibility assessment.

Given a trial (title, conditions, eligibility excerpt) and a patient note, decide: is this patient the KIND of patient this trial is for — the same primary condition or clinical area, and a compatible target population and setting?

Answer "no" ONLY for a categorical mismatch, e.g.:
- the patient's primary problem is fundamentally different from what the trial studies;
- the wrong anatomical site (a knee patient for a jaw-joint trial);
- the wrong target population for the trial's purpose (an adult for a pediatric trial; a patient who HAS the disease for a healthy-volunteer or prevention study);
- the patient lacks the trial's core diagnosis entirely.

A patient who has the right condition but would be EXCLUDED by some specific rule is STILL a candidate — answer "yes". When uncertain, answer "yes".

Respond with exactly one word: yes or no.""",

# v2_topical — the v1 "candidacy" framing over-drops rel=1: a rel=1 patient is INELIGIBLE by
# construction, so a candidacy judge rejects nearly all of them (v1 dropped 91% of rel=1). This
# reframes to pure TOPICAL relevance and states outright that ineligibility != irrelevance.
'v2_topical': """You are judging whether a clinical trial is TOPICALLY RELEVANT to a patient — the loose relevance a clinician uses when scanning trials for a patient, NOT whether the patient would be enrolled.

Given a trial (title, conditions, eligibility excerpt) and a patient note, answer "yes" if the trial studies the patient's condition, or a closely related one in the same clinical area / body system, such that this trial is worth considering for this patient.

CRITICAL: ineligibility does NOT make a trial irrelevant. Answer "yes" even if the patient would clearly be EXCLUDED — wrong age, wrong sex, declined the intervention, wrong disease stage, or fails any specific eligibility rule. A relevant-but-ineligible patient is a valid match, not a mismatch.

Answer "no" ONLY when the trial is about a FUNDAMENTALLY DIFFERENT condition or body system than the patient's — e.g. a knee-arthritis patient and a jaw-joint (temporomandibular) trial; a lymphoma patient and a diabetes trial; a patient who lacks an active disease that the trial specifically requires. A broad or vague trial condition label ("Healthy", "Infection", "Microbiota") does not by itself make an unrelated patient relevant — judge by the trial's actual subject in the title/eligibility.

Respond with exactly one word: yes or no.""",
}

# Note-style instructions (the {example} slot is filled with a real topic for register match)
STYLE_INSTR = {
    'trec_narrative': ('Write as a narrative case vignette, one or two paragraphs of flowing '
                       'clinical prose, in the style of this example:\n---\n{example}\n---'),
    'ehr_structured': ('Write as a structured EHR note with sections: HPI, PMH, Medications, '
                       'Social History, and Labs/Vitals where relevant. Telegraphic clinical style.'),
    'referral_brief': ('Write as a brief referral note, 3-5 sentences, terse.'),
}
print(f'registries: {len(MODELS)} models | gen {list(GEN_PROMPTS)} | verify {list(VERIFY_PROMPTS)} | '
      f'topic {list(TOPIC_PROMPTS)} | relevance {list(RELEVANCE_PROMPTS)}')

### `SynthConfig` — one frozen object; `tag()` names every output file

Two prompt/model variants never collide (each gets its own tagged files), and the downstream A/B can
attribute a result to the exact config that produced the data. **Flip `gen_prompt` / `gen_model` here.**

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class SynthConfig:
    data_root: str = '/content/drive/MyDrive/ct_data23'
    # --- models / prompts (keys into the registries) ---
    gen_model:    str = 'sonnet'
    verify_model: str = 'haiku'     # keep a DIFFERENT family from gen for an independent check
    qa_model:     str = 'opus'      # re-check for the base-disagrees QA queue (optional)
    gen_prompt:      str = 'v1'
    topic_prompt:    str = 'v1'
    verify_prompt:   str = 'v1'
    relevance_prompt: str = 'v2_topical'  # topical-relevance gate; v1 (candidacy) over-dropped rel=1 91%
    # --- sampling ---
    n_trials:      int = 1800
    n_per_trial:   int = 4
    rel_mix:   tuple = ((1, 0.6), (2, 0.4))   # rel=1 (scarce, load-bearing) oversampled
    style_mix: tuple = (('trec_narrative', 0.6), ('ehr_structured', 0.25), ('referral_brief', 0.15))
    max_inc: int = 4
    max_exc: int = 2
    min_crit_chars: int = 15
    max_crit_chars: int = 300
    min_usable_crit: int = 2
    # --- difficulty curation (base-monoT5 margin thresholds; DIAGNOSTIC + cap-easy) ---
    easy_margin:     float = 3.0    # margin >= this on a positive => base already correct (trivial)
    disagree_margin: float = -1.0   # margin <= this on a positive => base strongly disagrees => QA queue
    easy_keep_frac:  float = 0.30   # fraction of trivially-correct positives kept as anchors
    keep_hard:       bool  = False  # ⚙️ base_disagrees policy. False (v1/easy synth): drop them to QA
                                    # (base flatly rejecting a Haiku-passed positive = likely fidelity failure).
                                    # True (HARD synth, e.g. opus.v3_implicit): KEEP them — for genuinely-hard
                                    # notes base_disagrees are the VALUABLE hard positives (base can't read the
                                    # implicit evidence), not failures; they drive the develop ceiling up (§16).
    # --- infra ---
    concurrency: int = 12
    seed: int = 42
    base_monot5: str = 'castorini/monot5-3b-med-msmarco'  # difficulty scorer = the model we adapt

    def tag(self) -> str:
        return f'{self.gen_model}.{self.gen_prompt}__{self.verify_model}.{self.verify_prompt}'
    def notes_path(self)   -> str: return f'{self.data_root}/synth_notes__{self.tag()}.jsonl'
    def pairs_path(self)   -> str: return f'{self.data_root}/synth_pairs__{self.tag()}.jsonl'
    def curated_path(self) -> str: return f'{self.data_root}/synth_pairs__{self.tag()}__curated.jsonl'
    def qa_path(self)      -> str: return f'{self.data_root}/synth_qa_queue__{self.tag()}.jsonl'
    def relevance_path(self) -> str: return f'{self.data_root}/synth_relevance__{self.tag()}__{self.relevance_prompt}.jsonl'
    @property
    def canonical(self)    -> str: return f'{self.data_root}/synthetic_pairs.jsonl'  # consumers' default

# ⚙️ FULL OPUS RUN (§16 SOTA push): Opus is the confirmed hardness lever (70% easier-than-real vs
# sonnet 85% / v1 90%; equal-N probe showed harder data lifts develop). Resumes the earlier 1000-pair
# opus test (same tag) — skips the ~700 already done. keep_hard=True keeps the hard base_disagrees tail.
# 💵 COST: ~7,200 Opus generations @ max_tokens=2500 ≈ $200-250 + ~$5 Haiku verify. n_trials is the dial:
# drop to e.g. 1000 (→4,000 specs ≈ $120) if you want a smaller run — harder data is efficient (1,673
# beat 3,942), so you may not need the full set. STATE CHECK shows `planned` before you spend.
cfg = SynthConfig(gen_model='opus', gen_prompt='v3_implicit', keep_hard=True)   # n_trials=1800 default → 7,200 specs
import os
os.makedirs(cfg.data_root, exist_ok=True)
print('config tag:', cfg.tag())
print('  gen   :', MODELS[cfg.gen_model], '/', cfg.gen_prompt)
print('  verify:', MODELS[cfg.verify_model], '/', cfg.verify_prompt, '(+ topicality', cfg.topic_prompt + ')')
print('  pairs :', cfg.pairs_path())

## ▣ STATE CHECK — where am I?  ·  no API, no GPU; reads Drive and prints progress

Run this any time to see, for the **current `cfg.tag()`**, how far each phase has gotten. Safe to run
repeatedly; it only reads files. Tells you which phase to run next.

In [ ]:
def state_check():
    import os, json
    from collections import Counter
    def _cnt(p, pred=None):
        if not os.path.exists(p): return None
        n = 0
        for l in open(p):
            try: r = json.loads(l)
            except Exception: continue
            if pred is None or pred(r): n += 1
        return n
    # exact spec count once Phase 2 has run (some trials are skipped, so the n_trials*n_per_trial
    # product is only an upper bound — using it would make the 'incomplete' hint nag forever).
    planned   = len(globals()['all_specs']) if 'all_specs' in globals() else cfg.n_trials * cfg.n_per_trial
    notes_tot = _cnt(cfg.notes_path()); notes_ok = _cnt(cfg.notes_path(), lambda r: r.get('feasible'))
    ver_tot   = _cnt(cfg.pairs_path()); ver_ok = _cnt(cfg.pairs_path(), lambda r: r.get('verified'))
    curated   = _cnt(cfg.curated_path())
    canon_n   = (sum(1 for _ in open(cfg.canonical)) if os.path.exists(cfg.canonical) else None)
    def row(ok, phase, detail): print(f'  [{"✓" if ok else "○"}] {phase:<26s} {detail}')
    print(f'=== STATE for tag  {cfg.tag()}  (planned ≈ {planned:,} generations) ===')
    row(notes_tot, 'P4 generation',  f'{notes_tot or 0:,}/{planned:,} attempted · {notes_ok or 0:,} feasible notes' if notes_tot else 'not started')
    row(ver_tot,   'P5 verification', f'{ver_ok or 0:,} verified / {ver_tot or 0:,} judged' if ver_tot else 'not started')
    if ver_ok:
        by = Counter(json.loads(l)['rel'] for l in open(cfg.pairs_path()) if json.loads(l).get('verified'))
        print('        verified by rel:', dict(sorted(by.items())))
    row(curated,   'P7 curation',    f'{curated:,} curated training pairs' if curated else 'not run (needs GPU)')
    row(canon_n,   'P8 adopt',       f'{canon_n:,} rows in {os.path.basename(cfg.canonical)}' if canon_n else 'not adopted')
    # next-step hint
    if not notes_tot:                 nxt = 'run Phase 3 (pilot) then Phase 4 (generation)'
    elif (notes_tot or 0) < planned:  nxt = f'Phase 4 generation incomplete — re-run it (resumes; {planned-(notes_tot or 0):,} left)'
    elif not ver_tot or (ver_ok or 0) < (notes_ok or 0): nxt = 'run Phase 5 (verification)'
    elif not curated:                 nxt = 'run Phase 7 (difficulty + curation, GPU)'
    elif not canon_n:                 nxt = 'run Phase 8 (adopt)'
    else:                             nxt = 'done — adopt file ready for finetune_monot5_ct_v7'
    print('  → NEXT:', nxt)

state_check()

## PHASE 2 — Build specs  ·  corpus → trial pool → specs → prompts  ·  no API, no GPU

### Trial pool — sample from the corpus (representation-consistent)

Trials come from the corpus itself (`load_corpus`) so they already carry title/condition/summary/
eligibility in the frozen `R = elig_first-L512` schema and have a dense embedding (the judge mines against
it). inc/exc split with ctproc's `process_eligibility_naive`. **TREC21+22+23-judged trials are removed**
(the 21 exclusion closes the dev-arbiter confound).

In [ ]:
import json, random
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval
from ctproc.eligibility import process_eligibility_naive

xcfg = ExperimentConfig(data_root=cfg.data_root)

# contamination set: TREC21 + TREC22 (dev + test) judged trials, + TREC23 if the qrels file is present
contam, per_src = set(), {}
for src in ('trec21', 'trec22'):
    ds = load_eval(xcfg, [src]).get(src)
    ids = set()
    if ds:
        for d2r in ds['rel_dict'].values():
            ids |= set(d2r.keys())
    per_src[src] = len(ids); contam |= ids
QRELS_2023 = f'{cfg.data_root}/evaluation/trec_data/qrels2023.txt'
if os.path.exists(QRELS_2023):
    ids = set()
    for line in open(QRELS_2023):
        p = line.split()
        if len(p) >= 4: ids.add(p[2])
    per_src['trec23'] = len(ids - contam); contam |= ids
print('contamination guard (excluded from the synth pool):', per_src, '| total', len(contam))

# Narrative style few-shot = a few real TREC21 topic NOTES (register match). TREC21 is the go/no-go
# DEV set, so to keep that gate honest these donor topics are RECORDED and excluded from v7's dev+canary
# (v7 reads synth_style_donors.json). Only prose register is borrowed — synthetic conditions/labels are
# disjoint from the donors — but we exclude them anyway, for consistency with the trial-pool exclusion.
t21 = load_eval(xcfg, ['trec21'])['trec21']
STYLE_DONOR_K = 3
donor_ids = sorted(t21['topic2text'])[:STYLE_DONOR_K]
STYLE_DONORS = [t21['topic2text'][tid] for tid in donor_ids]
json.dump(donor_ids, open(f'{cfg.data_root}/synth_style_donors.json', 'w'))
print(f'style donors (recorded, excluded from v7 dev+canary): {donor_ids}')

In [ ]:
# --- load corpus, build the eligible trial pool ---
corpus_ids, corpus_fields = load_corpus(xcfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
print(f'corpus: {len(corpus_ids):,} trials')

def _first_condition(fields):
    c = fields.get('conditions')
    if isinstance(c, list):
        c = next((str(x).strip() for x in c if str(x).strip()), '')
    return (c or '').strip()

# Coerce a corpus field to a plain string (mirrors experiments._field_text): eligibility is normally a
# string, but guard against list/None so the splitter never sees a non-str.
def _as_text(v):
    if isinstance(v, list):
        return ' '.join(str(x) for x in v if x)
    return str(v) if v else ''

def usable(crit_list):
    return [c.strip() for c in crit_list
            if cfg.min_crit_chars <= len(c.strip()) <= cfg.max_crit_chars]

FIELD_KEYS = ('brief_title', 'official_title', 'conditions', 'brief_summary', 'detailed_desc', 'eligibility')

rng = random.Random(cfg.seed)
candidates = [d for d in corpus_ids if d not in contam]
rng.shuffle(candidates)

trial_pool, n_nocond, n_fewcrit = [], 0, 0
for d in candidates:
    if len(trial_pool) >= cfg.n_trials:
        break
    f = id2fields.get(d, {})
    cond = _first_condition(f)
    if not cond:
        n_nocond += 1; continue
    inc_raw, exc_raw = process_eligibility_naive(_as_text(f.get('eligibility')))
    inc, exc = usable(inc_raw), usable(exc_raw)
    if len(inc) + len(exc) < cfg.min_usable_crit:
        n_fewcrit += 1; continue
    trial_pool.append({'nct_id': d, 'condition': cond, 'inc': inc, 'exc': exc,
                       'fields': {k: f.get(k) for k in FIELD_KEYS}})

print(f'trial pool: {len(trial_pool)}  (skipped: no-condition {n_nocond}, <{cfg.min_usable_crit} crit {n_fewcrit})')
print(f'planned generations: {len(trial_pool) * cfg.n_per_trial:,}')

### Spec sampling — compose the graded label deterministically

Sample decisive per-criterion targets + an on-topic regime, then compose: `rel=2` ⟺ every criterion
satisfied; `rel=1` ⟺ exactly one disqualifier (an inclusion `not_included` or an exclusion `excluded`).
No `not_enough_information` targets, so the composed label is unambiguous. Deterministic under `cfg.seed`.

In [ ]:
from collections import Counter

def weighted_choice(pairs, r):
    return r.choices([k for k, _ in pairs], weights=[w for _, w in pairs], k=1)[0]

def sample_spec(trial, i, r):
    target_rel = weighted_choice(cfg.rel_mix, r)
    n_inc = r.randint(1, min(cfg.max_inc, len(trial['inc']))) if trial['inc'] else 0
    n_exc = r.randint(0, min(cfg.max_exc, len(trial['exc']))) if trial['exc'] else 0
    if n_inc + n_exc == 0:                       # ensure >=1 criterion, from a non-empty list
        if trial['inc']: n_inc = 1
        else:            n_exc = 1
    inc_sel = r.sample(trial['inc'], n_inc) if n_inc else []
    exc_sel = r.sample(trial['exc'], n_exc) if n_exc else []
    targets = ([{'crit_type': 'inclusion', 'criterion': c, 'label': 'included'} for c in inc_sel] +
               [{'crit_type': 'exclusion', 'criterion': c, 'label': 'not_excluded'} for c in exc_sel])
    if target_rel == 1:                          # flip exactly one criterion into a disqualifier
        j = r.randrange(len(targets))
        targets[j]['label'] = 'not_included' if targets[j]['crit_type'] == 'inclusion' else 'excluded'
    return {
        'syn_id': f"{trial['nct_id']}__{i}", 'nct_id': trial['nct_id'], 'condition': trial['condition'],
        'target_rel': target_rel, 'style': weighted_choice(cfg.style_mix, r),
        'age': r.randint(19, 84), 'sex': r.choice(['male', 'female']), 'targets': targets,
        'fields': trial['fields'],               # inline (already slimmed to FIELD_KEYS) -> self-contained
    }

all_specs = [sample_spec(t, i, rng) for t in trial_pool for i in range(cfg.n_per_trial)]
rel_dist = Counter(s['target_rel'] for s in all_specs)
print(f'specs: {len(all_specs):,}  ' + '  '.join(f'rel{k}={v} ({100*v/len(all_specs):.0f}%)' for k, v in sorted(rel_dist.items())))
s = all_specs[0]
print(f"\nsample: {s['syn_id']} rel={s['target_rel']} style={s['style']} cond={s['condition']}")
for t in s['targets']:
    print(f"  [{t['crit_type'][:3]}] {t['label']:14s} {t['criterion'][:66]}")

### Prompt assembly + parsing, then the generate/verify/compose coroutines (all keyed off `cfg`)

In [ ]:
import re

def make_gen_prompt(spec):
    style = STYLE_INSTR[spec['style']]
    if spec['style'] == 'trec_narrative':
        donor = STYLE_DONORS[int(spec['syn_id'].rsplit('__', 1)[-1]) % len(STYLE_DONORS)]  # rotate donors
        style = style.format(example=donor[:600])
    lines = [f"Patient: {spec['age']}-year-old {spec['sex']}",
             f"Condition (the patient clearly HAS this): {spec['condition']}",
             '', style, '', 'Criteria and target assessments:']
    for j, t in enumerate(spec['targets']):
        lines.append(f"{j}. [{t['crit_type']}] {t['criterion']}")
        lines.append(f"   TARGET: {t['label']}")
    return '\n'.join(lines)

VALID = {'included', 'not_included', 'excluded', 'not_excluded', 'not_enough_information'}
def extract_json(text):
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if not m: return None
    try:    return json.loads(m.group(0))
    except json.JSONDecodeError: return None
def parse_label(text):
    c = text.strip().lower().replace(' ', '_').strip('.')
    if c in VALID: return c
    for lab in sorted(VALID, key=len, reverse=True):
        if lab in c: return lab
    return 'not_enough_information'
def parse_yesno(text):
    t = text.strip().lower()
    return 'yes' if t.startswith('y') or 'yes' in t[:6] else 'no'

print(make_gen_prompt(all_specs[0]))

In [ ]:
# ── Async client + generate / verify coroutines (models & prompts from cfg) ───
import asyncio
from anthropic import AsyncAnthropic
aclient = AsyncAnthropic()

GEN_SYS    = GEN_PROMPTS[cfg.gen_prompt]
TOPIC_SYS  = TOPIC_PROMPTS[cfg.topic_prompt]
VERIFY_SYS = VERIFY_PROMPTS[cfg.verify_prompt]

async def gen_patient(spec, sem):
    async with sem:
        for attempt in range(3):
            try:
                resp = await aclient.messages.create(
                    model=MODELS[cfg.gen_model], max_tokens=2500, system=GEN_SYS,  # 2500: implicit/opus notes are long; 1500 truncated ~8% mid-JSON
                    messages=[{'role': 'user', 'content': make_gen_prompt(spec)}])
                # Fable 5 (and safety classifiers generally) can decline with a 200 + empty content.
                # Surface it distinctly and DON'T retry — the same prompt will refuse again.
                if resp.stop_reason == 'refusal' or not resp.content:
                    return spec, {'feasible': None, 'error': f'refusal (stop_reason={resp.stop_reason})'}
                data = extract_json(resp.content[0].text)
                if data is None:
                    return spec, {'feasible': None, 'error': 'unparseable JSON: ' + resp.content[0].text[:120]}
                return spec, data
            except Exception as e:
                if attempt == 2:
                    return spec, {'feasible': None, 'error': f'{type(e).__name__}: {e}'}
                await asyncio.sleep(2 ** attempt * 2)

async def verify_patient(rec, sem):
    """Topicality + per-criterion blind labels. The semaphore is acquired PER underlying call (not once
    per patient) so total concurrency stays bounded despite the 1 + n_criteria fan-out."""
    note = rec['patient_note']
    async def _topic():
        for attempt in range(3):
            try:
                async with sem:
                    r = await aclient.messages.create(
                        model=MODELS[cfg.verify_model], max_tokens=5, system=TOPIC_SYS,
                        messages=[{'role': 'user', 'content':
                            f"Patient: {note}\n\nCondition: {rec['condition']}\n\nDoes the patient have this condition?"}])
                return parse_yesno(r.content[0].text)
            except Exception:
                if attempt == 2: return None
                await asyncio.sleep(2 ** attempt * 2)
    async def _crit(t):
        msg = f"Patient: {note}\n\n{t['crit_type'].capitalize()} criterion: {t['criterion']}\n\nAssess this criterion."
        for attempt in range(3):
            try:
                async with sem:
                    r = await aclient.messages.create(
                        model=MODELS[cfg.verify_model], max_tokens=20, system=VERIFY_SYS,
                        messages=[{'role': 'user', 'content': msg}])
                return parse_label(r.content[0].text)
            except Exception:
                if attempt == 2: return None
                await asyncio.sleep(2 ** attempt * 2)
    topic, *crit_blinds = await asyncio.gather(_topic(), *[_crit(t) for t in rec['targets']])
    return rec, topic, crit_blinds

# Collapse the 4-way label to the clinical DECISION that composes rel. The verifier (and any reader)
# freely says "excluded" for an inclusion criterion the patient fails — colloquially "this disqualifies
# her" — where the strict label is "not_included". Those are the SAME decision, and rel only depends on
# the decision (satisfies vs disqualifies), so we verify on this axis, not the raw 4-way string. This
# recovers faithful rel=1 pairs lost to a labeling-taxonomy artifact, without accepting real misses
# (a satisfies/disqualifies flip, or an NEI, still fails).
def _decision(label):
    if label in ('included', 'not_excluded'):  return 'satisfies'
    if label in ('not_included', 'excluded'):  return 'disqualifies'
    return 'nei'                                 # not_enough_information

def compose(rec, topic, crit_blinds):
    targets = rec['targets']
    agreements = [{'criterion': t['criterion'], 'crit_type': t['crit_type'],
                   'target': t['label'], 'blind': b,
                   'match': (b is not None and _decision(b) == _decision(t['label']))}
                  for t, b in zip(targets, crit_blinds)]
    disqual = any(t['label'] in ('not_included', 'excluded') for t in targets)
    topic_ok = (topic == 'yes')
    crit_ok  = all(a['match'] for a in agreements)
    return {
        'syn_id': rec['syn_id'], 'nct_id': rec['nct_id'], 'condition': rec['condition'],
        'style': rec['style'], 'rel': 1 if disqual else 2, 'patient_note': rec['patient_note'],
        'fields': rec['fields'], 'targets': targets, 'topicality': topic,
        'criterion_agreement': agreements, 'verified': bool(topic_ok and crit_ok),
        'gen_tag': cfg.tag(),                    # provenance: which config produced this row
    }
print('coroutines ready | gen', MODELS[cfg.gen_model], '| verify', MODELS[cfg.verify_model])

## PHASE 3 — Diagnostic + pilot  ·  ~$0.20 API  ·  read the reports before the full run

### Connectivity diagnostic — run this FIRST if the pilot returns n=0

Makes ONE raw call to the generator model and ONE to the verifier, with **no retry and no
error-swallowing**, and prints the exact `stop_reason`, content, and any exception. This is the fast
way to see *why* calls fail (auth, a refusal, a data-retention 400, a bad model id) instead of a silent
zero. `n=0` from the pilot almost always means every generation errored and was filtered out.

In [ ]:
async def diagnose():
    for role, model in [('GEN', MODELS[cfg.gen_model]), ('VERIFY', MODELS[cfg.verify_model])]:
        print(f'\n=== {role}: {model} ===')
        try:
            resp = await aclient.messages.create(
                model=model, max_tokens=64,
                messages=[{'role': 'user', 'content': 'Reply with the single word: ok'}])
            print('  ok      | stop_reason =', resp.stop_reason, '| content blocks =', len(resp.content))
            if resp.stop_reason == 'refusal' or not resp.content:
                print('  ⚠ REFUSAL/EMPTY — safety classifier declined even a trivial prompt. On Fable 5 this'
                      '\n    hits life-sciences content; switch cfg.gen_model to "sonnet", or enable fallbacks (below).')
            else:
                print('  text    =', repr(resp.content[0].text[:80]))
        except Exception as e:
            print(f'  ✗ {type(e).__name__}: {e}')
            m = str(e).lower()
            if 'authentication' in m or '401' in m:  print('    → ANTHROPIC_API_KEY is missing/invalid (cell-apikey).')
            elif 'not_found' in m or '404' in m:     print(f'    → model id "{model}" not available to this key.')
            elif 'retention' in m or ('400' in m and 'fable' in model): print('    → Fable 5 needs 30-day data retention; a ZDR org 400s every request. Use "sonnet".')
            elif 'permission' in m or '403' in m:    print(f'    → this key lacks access to "{model}".')
            elif 'rate' in m or '429' in m:          print('    → rate limited; lower cfg.concurrency and retry.')

# One real generation, un-swallowed, so you see a full patient note or the exact failure:
print('--- trivial connectivity ---')
await diagnose()
print('\n--- one real generation (spec 0) ---')
_spec, _data = await gen_patient(all_specs[0], asyncio.Semaphore(1))
print('feasible:', _data.get('feasible'), '| error:', _data.get('error'))
if _data.get('feasible'):
    print(_data['patient_note'][:300])

### (optional) Enable server-side refusal fallbacks for Fable 5

If you keep `fable` as the generator and see refusals above, wrap generation so a declined request is
transparently re-served by Opus 4.8 in the same call. Only needed for `fable`; skip for `sonnet`.

In [ ]:
# Only meaningful when cfg.gen_model == 'fable'. Rebinds gen_patient to use the beta fallbacks param.
if cfg.gen_model == 'fable':
    async def gen_patient(spec, sem):                       # noqa: F811 (intentional override)
        async with sem:
            for attempt in range(3):
                try:
                    resp = await aclient.beta.messages.create(
                        model='claude-fable-5', max_tokens=1500, system=GEN_SYS,
                        betas=['server-side-fallback-2026-06-01'],
                        fallbacks=[{'model': 'claude-opus-4-8'}],
                        messages=[{'role': 'user', 'content': make_gen_prompt(spec)}])
                    if resp.stop_reason == 'refusal' or not resp.content:
                        return spec, {'feasible': None, 'error': f'refusal after fallback (stop_reason={resp.stop_reason})'}
                    data = extract_json(resp.content[0].text)
                    if data is None:
                        return spec, {'feasible': None, 'error': 'unparseable JSON'}
                    return spec, data
                except Exception as e:
                    if attempt == 2: return spec, {'feasible': None, 'error': f'{type(e).__name__}: {e}'}
                    await asyncio.sleep(2 ** attempt * 2)
    print('gen_patient now uses Fable 5 + server-side fallback to Opus 4.8 on refusal')
else:
    print(f'cfg.gen_model={cfg.gen_model!r} (not fable) — no fallback wrapper needed')

### Pilot — 60 patients through the full generate→verify loop + model-free triage

Read the per-class agreement matrix **and** the model-free triage (distinct conditions, exact-duplicate
rate) before spending the budget. rel=1 is the MVP and the most fragile. Does not write production files.

In [ ]:
def agreement_report(rows, label=''):
    print(f'\n=== agreement {label} (n={len(rows)}) ===')
    by = Counter(r['rel'] for r in rows); ok = Counter(r['rel'] for r in rows if r['verified'])
    tok = sum(1 for r in rows if r['topicality'] == 'yes')
    print(f'topicality pass: {tok}/{len(rows)} ({100*tok/max(len(rows),1):.0f}%)')
    for rel in sorted(by):
        print(f'  rel={rel} verified: {ok[rel]:4d}/{by[rel]:4d} ({100*ok[rel]/by[rel]:.0f}%)')
    # decision-collapsed accounting (see _decision): recovered = raw-label mismatch but same decision.
    aa = [a for r in rows for a in r['criterion_agreement']]
    recovered = Counter((a['target'], a['blind']) for a in aa if a.get('match') and a['blind'] != a['target'])
    genuine   = Counter((a['target'], a['blind']) for a in aa if not a.get('match'))
    if recovered:
        print(f'  taxonomy-recovered (same decision, different label): {sum(recovered.values())}')
        for (tg, bl), c in recovered.most_common(4):
            print(f'    ~ {tg:14s} == {str(bl):22s} {c}')
    for (tg, bl), c in genuine.most_common(6):
        print(f'  disagree {tg:14s} -> {str(bl):22s} {c}')

def triage_report(rows, label=''):   # model-free, cheap — catches mode collapse before you pay more
    notes = [r['patient_note'] for r in rows if r['verified']]
    norm = [' '.join(n.lower().split()) for n in notes]
    dup = len(norm) - len(set(norm))
    conds = Counter(r['condition'] for r in rows if r['verified'])
    lens = sorted(len(n.split()) for n in notes)
    print(f'--- triage {label}: verified {len(notes)} | distinct conditions {len(conds)} | '
          f'exact-dup notes {dup} | note words p50={lens[len(lens)//2] if lens else 0}')

async def run_pilot(specs):
    sem = asyncio.Semaphore(cfg.concurrency)
    gens = await asyncio.gather(*[gen_patient(s, sem) for s in specs])
    recs = [{**spec, 'patient_note': d['patient_note']} for spec, d in gens if d.get('feasible')]
    # Surface WHY generations failed instead of silently dropping them (the n=0 trap).
    errs = [d['error'] for _, d in gens if d.get('feasible') is None]         # API/refusal/parse errors
    infeasible = sum(1 for _, d in gens if d.get('feasible') is False)         # model said spec contradictory
    if errs or not recs:
        from collections import Counter as _C
        kinds = _C(e.split(':')[0] for e in errs)
        print(f'⚠ generation: {len(recs)} ok | {infeasible} infeasible(spec) | {len(errs)} errored → {dict(kinds)}')
        for e in errs[:3]:
            print('   e.g.', e[:160])
        if not recs:
            print('   → 0 usable notes. Run the diagnostic cell above to see the raw failure.')
    vers = await asyncio.gather(*[verify_patient(r, sem) for r in recs])
    return [compose(r, topic, cb) for r, topic, cb in vers]

pilot = await run_pilot(all_specs[:60])
agreement_report(pilot, 'PILOT')
triage_report(pilot, 'PILOT')
for r in pilot:
    if r['verified'] and r['rel'] == 1:
        print(f"\n--- verified rel=1 example [{r['condition']}] ---\n{r['patient_note'][:380]}...")
        for a in r['criterion_agreement']:
            print(f"  {a['target']:14s} == {a['blind']:14s}  {a['criterion'][:56]}")
        break

## PHASE 4 — Full generation → `notes_path`  ·  ~$30–50 API  ·  ✅ resumes by `syn_id`

The loop is wrapped in `run_generation(all_specs)`; the cell body is one call. Re-run the cell any time —
it skips already-generated `syn_id`s and only pays for what's left.

In [ ]:
from tqdm.auto import tqdm

async def run_generation(specs):
    NOTES = cfg.notes_path()
    done_gen = {json.loads(l)['syn_id'] for l in open(NOTES)} if os.path.exists(NOTES) else set()
    todo = [s for s in specs if s['syn_id'] not in done_gen]
    print(f'generated: {len(done_gen):,} | remaining: {len(todo):,} -> {NOTES}')

    sem = asyncio.Semaphore(cfg.concurrency)
    n_ok = n_inf = n_fail = 0
    err_samples = Counter()
    for start in tqdm(range(0, len(todo), 100), desc='generating'):
        chunk = todo[start:start+100]
        results = await asyncio.gather(*[gen_patient(s, sem) for s in chunk])
        with open(NOTES, 'a') as out:
            for spec, data in results:
                if data.get('feasible') is None:
                    n_fail += 1; err_samples[data.get('error', '?').split(':')[0]] += 1; continue
                if not data['feasible']:
                    n_inf += 1; out.write(json.dumps({'syn_id': spec['syn_id'], 'feasible': False}) + '\n'); continue
                n_ok += 1
                out.write(json.dumps({**spec, 'feasible': True, 'patient_note': data['patient_note']}) + '\n')
    print(f'\nok {n_ok:,} | infeasible {n_inf:,} | failed(retry) {n_fail:,}')
    if n_fail:
        print('  failure reasons:', dict(err_samples))
        if n_ok == 0:
            print('  ⚠ 0 succeeded — SYSTEMIC failure (not per-spec). Run `await diagnose()`: usually a rotated/dead'
                  '\n    key (401 → fix the API-key cell), or quota/credit exhaustion (429/billing). Verification hits'
                  '\n    the same API, so fix this before running it.')
    _feas_total = sum(1 for l in open(NOTES) if json.loads(l).get('feasible'))
    print(f'usable feasible notes in file: {_feas_total:,}')

await run_generation(all_specs)

## PHASE 5 — Verification → `pairs_path`  ·  ~$5 API  ·  ✅ resumes by `syn_id`

`run_verification()` runs the topicality + per-criterion blind checks and writes verified pairs; the
fidelity-report cell after it is the dataset card. Same API as generation — if generation was failing
(bad key / quota), fix that first.

In [ ]:
async def run_verification():
    NOTES = cfg.notes_path()
    patients = {}
    if os.path.exists(NOTES):
        for l in open(NOTES):
            rec = json.loads(l)
            if rec.get('feasible'): patients[rec['syn_id']] = rec
    PAIRS = cfg.pairs_path()
    done_ver = {json.loads(l)['syn_id'] for l in open(PAIRS)} if os.path.exists(PAIRS) else set()
    todo_v = [p for p in patients.values() if p['syn_id'] not in done_ver]
    print(f'feasible {len(patients):,} | verified {len(done_ver):,} | remaining {len(todo_v):,} -> {PAIRS}')

    if not patients:
        print('⚠ 0 feasible notes — nothing to verify. Run Phase 4 generation (and `await diagnose()`) first.')
        return
    vsem = asyncio.Semaphore(cfg.concurrency)
    n_written = n_apifail = 0
    for start in tqdm(range(0, len(todo_v), 100), desc='verifying'):
        chunk = todo_v[start:start+100]
        results = await asyncio.gather(*[verify_patient(r, vsem) for r in chunk])
        with open(PAIRS, 'a') as out:
            for rec, topic, cb in results:
                if topic is None or any(b is None for b in cb):
                    n_apifail += 1; continue      # API failure -> retry next run (same-API issue as generation)
                out.write(json.dumps(compose(rec, topic, cb)) + '\n'); n_written += 1
    print(f'wrote {n_written:,} verified-attempt records | {n_apifail:,} API-failed (will retry)')
    if todo_v and n_written == 0:
        print('  ⚠ 0 written despite feasible notes — verification API calls are failing. Same fix as generation:'
              ' run `await diagnose()` (rotated key / quota) before retrying.')
    print('done')

await run_verification()

### Fidelity report (the dataset card)

In [ ]:
rows = [json.loads(l) for l in open(cfg.pairs_path())]
verified = [r for r in rows if r['verified']]
print(f'records {len(rows):,} | kept (verified) {len(verified):,} ({100*len(verified)/max(len(rows),1):.1f}%)')
agreement_report(rows, 'FULL'); triage_report(rows, 'FULL')
print('verified by rel:', {k: v for k, v in sorted(Counter(r['rel'] for r in verified).items())})
print('style mix:', dict(Counter(r['style'] for r in verified)), '| distinct trials', len({r['nct_id'] for r in verified}))

## PHASE 6 — Whole-trial relevance gate  ·  ⚙️ DISABLED by default (documented negative result)

The idea: `rel` is a *whole-trial* claim, but the per-criterion verifier only checks *sampled* criteria,
so a patient can satisfy one cherry-picked criterion while being the wrong kind of patient for the trial
(knee patient → jaw-joint trial). Reading the QA queue confirmed such mislabels are real. **But two gate
prompts both failed**: `v1` (candidacy) dropped **91% of rel=1**, `v2_topical` (topical, ineligibility≠
irrelevance) still dropped **84%**. The cause is structural — a rel=1 patient is *built* to carry a
disqualifying feature, so any relevance judge scores it down; no prompt separates "relevant-but-
ineligible" (a valid rel=1) from "irrelevant". And it is **unnecessary for the primary consumer**:
monoT5 v7 is pointwise (`rel≥1 → 'true'`), so a valid rel=1 is a *correct* positive — dropping it removes
good signal — while the harmful mislabels it targets are either already caught by the difficulty
curation's `base_disagrees` bucket or are low-gradient (monoT5 already scores them plausibly). Net-
negative for monoT5. **Default `RUN_RELEVANCE_GATE=False`; the v7 A/B + canary arbitrate residual noise.**
Re-enable only for the judge (graded, listwise) if a future, non-conflating design exists (e.g. a
condition-only topic match, or Fix B: include core inclusions in the spec at generation time).

In [ ]:
from tqdm.auto import tqdm
from collections import Counter
RELEVANCE_SYS = RELEVANCE_PROMPTS[cfg.relevance_prompt]

def trial_summary(f):
    t = f.get('official_title') or f.get('brief_title') or ''
    c = f.get('conditions'); c = ', '.join(str(x) for x in c if x) if isinstance(c, list) else (c or '')
    e = (f.get('eligibility') or '')[:700]
    return f'Title: {t}\nConditions: {c}\nEligibility (excerpt): {e}'

async def relevance_gate(rec, sem):
    msg = (f"TRIAL:\n{trial_summary(rec['fields'])}\n\nPATIENT:\n{rec['patient_note']}\n\n"
           'Is this patient a plausible candidate for THIS trial?')
    for attempt in range(3):
        try:
            async with sem:
                r = await aclient.messages.create(model=MODELS[cfg.verify_model], max_tokens=5,
                    system=RELEVANCE_SYS, messages=[{'role': 'user', 'content': msg}])
            return rec['syn_id'], parse_yesno(r.content[0].text)
        except Exception:
            if attempt == 2: return rec['syn_id'], None
            await asyncio.sleep(2 ** attempt * 2)

# DISABLED: both gate prompts over-dropped rel=1 (v1 candidacy 91%, v2_topical 84%) — a rel=1 patient is
# ineligible by construction, so any relevance judge culls valid rel=1. For pointwise monoT5 a valid
# rel=1 IS a correct positive. Flip to True only to reproduce the negative result or gate the judge.
RUN_RELEVANCE_GATE = False

if not RUN_RELEVANCE_GATE:
    print('relevance gate DISABLED — the difficulty cell will train on the full verified+curated set '
          '(rel=1 preserved). The v7 A/B + canary arbitrate residual mislabel noise.')
else:
    verified = [json.loads(l) for l in open(cfg.pairs_path()) if json.loads(l).get('verified')]
    REL = cfg.relevance_path()
    done = {json.loads(l)['syn_id'] for l in open(REL)} if os.path.exists(REL) else set()
    todo = [r for r in verified if r['syn_id'] not in done]
    print(f'verified pairs {len(verified)} | relevance-judged {len(done)} | remaining {len(todo)} -> {REL}')
    sem = asyncio.Semaphore(cfg.concurrency)
    n_fail = 0
    for start in tqdm(range(0, len(todo), 100), desc='relevance gate'):
        chunk = todo[start:start+100]
        res = await asyncio.gather(*[relevance_gate(r, sem) for r in chunk])
        with open(REL, 'a') as out:
            for sid, ans in res:
                if ans is None: n_fail += 1; continue
                out.write(json.dumps({'syn_id': sid, 'relevant': ans == 'yes'}) + '\n')
    rel_map = {json.loads(l)['syn_id']: json.loads(l)['relevant'] for l in open(REL)}
    judged  = [r for r in verified if r['syn_id'] in rel_map]
    dropped = [r for r in judged if not rel_map[r['syn_id']]]
    byrel, dbyrel = Counter(r['rel'] for r in judged), Counter(r['rel'] for r in dropped)
    print(f'\nrelevance gate: {len(judged)} judged | {len(dropped)} dropped '
          f'({100*len(dropped)/max(len(judged),1):.1f}%)' + (f' | {n_fail} api-failed(retry)' if n_fail else ''))
    for rel in sorted(byrel):
        print(f'  rel={rel}: dropped {dbyrel[rel]}/{byrel[rel]} ({100*dbyrel[rel]/max(byrel[rel],1):.0f}%)')
    print('  ⚠ if rel=1 drop ≫ rel=2 drop, the gate is culling valid rel=1 — leave it disabled.')

## PHASE 7 — Difficulty diagnostic + real-TREC21 calibration + curation  ·  **GPU** (base monoT5-3B)

**This is the pre-spend gate on "is the generator good enough?"** It scores every verified synth
positive with **base monoT5-MED** (`cfg.base_monot5` — the exact model v7 adapts) *and* a sample of **real
TREC21 judged pairs** with the same model, then overlays the margin distributions. The question the
histogram answers: **are our synth positives as hard for the base model as real TREC21 positives, or
systematically easier?** Fidelity (blind-verify) can be high while notes are trivially easy — evidence
spelled out too explicitly — which gives a weak training gradient and overfits the template (no transfer).
The verdict line reports **% of synth positives easier than the median real positive**: ~50% = matched
hardness (good); ≫50% = too easy (tighten the prompt to `v2_strict`, or use a stronger generator).

Set `SYNTH_SOURCE='pilot'` to run this on the ~40 in-memory pilot pairs **before** the full generation
(GPU runtime required — run the pilot, then this). Set `'file'` after the full run to also **curate**:
- **cap** trivially-correct positives (`margin >= easy_margin`) to `easy_keep_frac` (low gradient; a sample
  kept as anti-forgetting anchors); **keep** the mid band; **route** base-strongly-disagrees positives
  (`margin <= disagree_margin`) to a **QA queue**, not training — for a positive, monoT5 flatly disagreeing
  while the verifier passed it is disproportionately a *fidelity* failure, and auto-keeping it injects label
  noise into the high-gradient (forgetting) region. We do NOT maximize "hard"; the v7 **canary** arbitrates.

The real-TREC21 pairs are a **calibration reference only** (never training/dev) — scoring them here is not
a leak. Writes `curated_path` (+ `base_margin`/`bucket`/`curate`) and `qa_path` in `file` mode.

In [ ]:
import torch, numpy as np, random as _r
from transformers import T5Tokenizer, T5ForConditionalGeneration

# auto: 'file' once the full pairs file exists (curates the whole set), else 'pilot' (pre-spend check on
# the in-memory pilot). Override manually to force either. This makes a top-to-bottom "Run All" do the
# right thing in both states.
SYNTH_SOURCE = 'file' if os.path.exists(cfg.pairs_path()) else 'pilot'
if SYNTH_SOURCE == 'pilot':
    verified = [r for r in pilot if r['verified']]
else:
    verified = [json.loads(l) for l in open(cfg.pairs_path()) if json.loads(l).get('verified')]
# Relevance gate is DISABLED by default (over-drops rel=1; see the relevance cell). Keep this in sync
# with RUN_RELEVANCE_GATE. When off, we train on the full verified+curated set (rel=1 preserved).
APPLY_RELEVANCE_GATE = False
if APPLY_RELEVANCE_GATE and os.path.exists(cfg.relevance_path()):
    _relmap = {json.loads(l)['syn_id']: json.loads(l)['relevant'] for l in open(cfg.relevance_path())}
    _b = len(verified); verified = [r for r in verified if _relmap.get(r['syn_id'], True)]
    print(f'relevance gate applied: dropped {_b - len(verified)}')
print(f'diagnosing {len(verified)} synth positives (source={SYNTH_SOURCE}) with base {cfg.base_monot5}')

def _doc_str(f, with_desc=False):
    t = f.get('brief_title') or f.get('official_title') or ''
    c = f.get('conditions', '') or ''
    if isinstance(c, list): c = ', '.join(str(x) for x in c if x)
    e = f.get('eligibility', '') or ''
    s = f"title: {t} condition: {c} eligibility: {e}"
    if with_desc:
        s += f" description: {f.get('detailed_desc') or f.get('brief_summary') or ''}"
    return s[:1400]

_tok = T5Tokenizer.from_pretrained(cfg.base_monot5)
_m = T5ForConditionalGeneration.from_pretrained(cfg.base_monot5, torch_dtype=torch.float16, device_map='auto').eval()
_T = _tok('true', add_special_tokens=False).input_ids[0]; _F = _tok('false', add_special_tokens=False).input_ids[0]

@torch.no_grad()
def base_margins(pairs, b=16):
    out = []
    for i in range(0, len(pairs), b):
        ins = [f"Query: {r['patient_note']} Document: {_doc_str(r['fields'])} Relevant:" for r in pairs[i:i+b]]
        enc = _tok(ins, return_tensors='pt', padding=True, truncation=True, max_length=512).to(_m.device)
        dec = torch.zeros((enc['input_ids'].shape[0], 1), dtype=torch.long, device=_m.device)
        lp = torch.log_softmax(_m(**enc, decoder_input_ids=dec).logits[:, 0, :].float(), -1)
        out.extend((lp[:, _T] - lp[:, _F]).cpu().tolist())
    return out

# ── real TREC21 reference pairs (topic note + judged trial fields) — CALIBRATION only, never train/dev ──
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval
_xc = ExperimentConfig(data_root=cfg.data_root)
_cids, _cfs = load_corpus(_xc); _id2f = dict(zip(_cids, _cfs))
_s21 = load_eval(_xc, ['trec21'])['trec21']; _rel21 = _s21['rel_dict']; _t2t = _s21['topic2text']
REF_CAP = 400
_pos_ref, _neg_ref, _rr = [], [], _r.Random(cfg.seed)
for _tid, _d2r in _rel21.items():
    _qt = _t2t.get(_tid)
    if not _qt: continue
    for _d, _g in _d2r.items():
        if _d in _id2f:
            (_pos_ref if _g >= 1 else _neg_ref).append({'patient_note': _qt, 'fields': _id2f[_d]})
_rr.shuffle(_pos_ref); _rr.shuffle(_neg_ref)
_pos_ref, _neg_ref = _pos_ref[:REF_CAP], _neg_ref[:REF_CAP]
print(f'real TREC21 reference: {len(_pos_ref)} positives, {len(_neg_ref)} negatives')

m_syn  = base_margins(verified)
m_rpos = base_margins(_pos_ref)
m_rneg = base_margins(_neg_ref)
for r, mg in zip(verified, m_syn):
    r['base_margin'] = float(mg)
    r['bucket'] = ('trivially_correct' if mg >= cfg.easy_margin
                   else 'base_disagrees' if mg <= cfg.disagree_margin else 'mid')

def _pct(x):
    x = np.asarray(x)
    return f'mean {x.mean():+.2f} | p10 {np.percentile(x,10):+.2f} | p50 {np.percentile(x,50):+.2f} | p90 {np.percentile(x,90):+.2f}'
def _hist(x, lo=-15, hi=10, w=26):
    x = np.clip(np.asarray(x), lo, hi); edges = np.linspace(lo, hi, w + 1)
    h, _ = np.histogram(x, bins=edges); top = max(h.max(), 1)
    return '\n'.join(f'    {edges[i]:+5.0f} | {"#" * int(22 * h[i] / top)} {h[i]}' for i in range(w) if h[i])

print('\n── base-monoT5 margin: synth positives vs REAL TREC21 (higher = base thinks MORE relevant = easier) ──')
print(f'  synth positives (n={len(m_syn)}):        {_pct(m_syn)}')
print(f'  real TREC21 positives (n={len(m_rpos)}): {_pct(m_rpos)}')
print(f'  real TREC21 negatives (n={len(m_rneg)}): {_pct(m_rneg)}')
_rpos_med = float(np.percentile(m_rpos, 50))
_easier = float(np.mean(np.asarray(m_syn) > _rpos_med))
print(f'\n  VERDICT: {100*_easier:.0f}% of synth positives are EASIER than the median real TREC21 positive '
      f'(margin > {_rpos_med:+.2f}).')
print('    ~50% = synth matches real hardness (GOOD — proceed). >>50% = synth systematically too easy')
print('    (weak gradient + transfer risk) -> switch cfg.gen_prompt to "v2_strict" or a stronger cfg.gen_model.')
print('\n  synth-positive margins:');       print(_hist(m_syn))
print('  real-TREC21-positive margins:');   print(_hist(m_rpos))
print('\n  buckets (cfg thresholds):', dict(Counter(r['bucket'] for r in verified)))

if SYNTH_SOURCE == 'file':   # curation writes the training file only on the full run
    crng = _r.Random(cfg.seed); kept = qa = hard_kept = 0
    with open(cfg.curated_path(), 'w') as cf, open(cfg.qa_path(), 'w') as qf:
        for r in verified:
            if r['bucket'] == 'base_disagrees' and not cfg.keep_hard:
                r['curate'] = False; qf.write(json.dumps(r) + '\n'); qa += 1   # v1/easy: drop to QA
            elif r['bucket'] == 'trivially_correct' and crng.random() > cfg.easy_keep_frac:
                r['curate'] = False                                            # cap trivially-easy
            else:
                r['curate'] = True; cf.write(json.dumps(r) + '\n'); kept += 1  # keep (base_disagrees too, if keep_hard)
                if r['bucket'] == 'base_disagrees': hard_kept += 1
    print(f'\ncurated (train) {kept} -> {cfg.curated_path()}'
          + (f'  [incl {hard_kept} base_disagrees KEPT as hard positives — keep_hard=True]' if cfg.keep_hard else ''))
    print(f'QA queue (re-inspect w/ {MODELS[cfg.qa_model]}) {qa} -> {cfg.qa_path()}'
          + ('  [empty: keep_hard=True routes base_disagrees to training]' if cfg.keep_hard else ''))
else:
    print('\n(pilot source: diagnostic only — no training file written. Use SYNTH_SOURCE="file" after the full run.)')
del _m; torch.cuda.empty_cache()

## PHASE 8 — Adopt → the canonical file the consumers read

Consumers (`finetune_monot5_ct_v7`, `finetune_judge_lora`) default `SYNTH_PAIRS_PATH` to
`cfg.canonical`. Copy whichever variant won the downstream A/B (curated is the smart default; use
`pairs_path()` to train on all verified). Then run v7 twice — `SYNTH_PAIRS_PATH=''` vs this file — and
compare TREC21 dev NDCG + canary retention (v7 logs the A/B).

In [ ]:
import shutil
SRC = cfg.curated_path() if os.path.exists(cfg.curated_path()) else cfg.pairs_path()
shutil.copy(SRC, cfg.canonical)
n = sum(1 for _ in open(cfg.canonical))
print(f'adopted {SRC}\n     -> {cfg.canonical}  ({n:,} rows)')
print('\nnext: in finetune_monot5_ct_v7.ipynb run KZ-only (SYNTH_PAIRS_PATH=\'\') then KZ+synth;')
print('compare TREC21 dev NDCG (bang-for-buck) + canary-gap retention (no forgetting).')